<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week9_data_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** These exercises are meant to be worked through yourself – treat this the same as switching off a calculator's solver mode in an exam. It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

# Week 9 – Data Exercises: Causal Analysis and Experiments

Adapted from Bekes & Kezdi, *Data Analysis for Business, Economics, and Policy* – the textbook this module follows, drawing on both the causal-framework chapter (Ch19) and the experiments chapter (Ch20). No output shown here to check yourself against: the point is to practise drawing causal maps, reasoning about confounders vs. mediators, and interpreting experimental results, on data and questions you haven't seen the answer for.

Easier and/or shorter exercises are marked **[\*]**; harder and/or longer exercises are marked **[\*\*]**.

Two of the textbook's own exercises for this chapter were left out deliberately: one needs a piecewise linear spline (that's Week 5's functional-form content, not this week's), and one revisits the electricity/temperature case study (that's Week 11's dataset) – both are more a test of earlier weeks' skills than this week's own causal-reasoning content, so they're better candidates for a later review exercise than for here.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower

food_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/food_health.csv"
share_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/share_health.csv"
wfh_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/wfh_person.csv"
ab_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/ab_test_social_media.csv"

food = pd.read_csv(food_path)
share = pd.read_csv(share_path)
wfh = pd.read_csv(wfh_path)
ab = pd.read_csv(ab_path)
food.head()

> **Syntax hint – a publication-style comparison table.** Whenever a question below asks you to compare models, present them side by side rather than just quoting numbers in prose:
>
> ```python
> def stars(p):
>     return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
>
> def compare_models(models, names, variables):
>     rows = []
>     for param_name, label in variables:
>         row = {'Variable': label}
>         for name, m in zip(names, models):
>             row[name] = f"{m.params[param_name]:.3f}{stars(m.pvalues[param_name])}" if param_name in m.params.index else ''
>         rows.append(row)
>     rows.append({'Variable': 'N', **{name: str(int(m.nobs)) for name, m in zip(names, models)}})
>     return pd.DataFrame(rows)
>
> print(compare_models([model_a, model_b], ['(1)', '(2)'], [('treatment', 'Treatment')]).to_markdown(index=False))
> ```

## Question 1 [\*]

Use `food` (the NHANES food-health data) and estimate a regression of `blood_pressure` on `veggies_n_fruits_gr` separately among smokers (`smoker == 1`) and non-smokers (`smoker == 0`).

1. Fit both regressions and present them in a comparison table.
2. Compare the results and explain why they might be different – what could smoking status be picking up here, beyond just smoking itself?

## Question 2 [\*]

Use `food` and construct a measure of the amount of unhealthy food consumed, using some of the food items in the data (candidates: `gr_candy`, `gr_soda`, `gr_sugar`, `gr_corn_based_chips`, `gr_sugar_cereal`, `gr_breakfast_pastry`, `gr_fruit_juice_drink`, `gr_potato_chips`). Explain your choices.

1. Investigate the association of your unhealthy-food measure with `blood_pressure`.
2. Investigate its association with `veggies_n_fruits_gr`.
3. Discuss whether, and why, you would (or wouldn't) want to condition on your unhealthy-food measure when estimating the effect of fruit-and-vegetable consumption on blood pressure. Is it a confounder, a mediator, or something else?

*Your reasoning on whether to condition on the unhealthy-food measure:*

## Question 3 [\*\*]

Consider the Week 8 case study on **does smoking pose a health risk?**, using `share` (the SHARE health-survey data).

1. Define the effect of current smoking on staying healthy, as it's measured in that case study.
2. Write down the sources of variation in `smoking` – why do some people in this data smoke and others don't? Draw a causal map (on paper, or as a comment in a code cell) and explain what's on it.
3. The data contains variables on income (`income10`), exercising (`exerc`), and education (`eduyears`). Can you use these to measure some of the confounders you identified? Can you measure them perfectly?
4. Discuss what your findings imply for whether, and how much, we can credibly estimate the effect of smoking on staying healthy using this data.
5. Are the results in the Week 8 case study in line with what's found in the wider literature on the health effects of smoking? What's your conclusion?

*Your causal map, and your conclusion about what can (and can't) be credibly estimated here:*

## Question 4 [\*]

Use `wfh` (the Working from Home RCT data) and restrict to `ordertaker == True` – the same subgroup the workshop's phone-calls result used.

1. Estimate the effect of `treatment` on `phonecalls1` among employees who **didn't quit** (`quitjob == 0`).
2. Compare this to the effect estimated on the **full** order-taker sample (i.e. not restricting by `quitjob`) – put both in a comparison table.
3. Interpret both coefficients, and explain what the difference between them means.

## Question 5 [\*]

Use `wfh`, restricted to order-takers again.

1. Estimate the effect of `treatment` on `phonecalls1`, including the number of phone calls in the pre-treatment period (`phonecalls0`) as a control.
2. Now also include the other pre-treatment covariates available (`age`, `costofcommute`, `children`, `male`, `married`, `prior_experience`, `tenure`, `rental`, `bedroom`, `second_technical`, `high_school`, `tertiary_technical`, `university`, `internet`) – **except** `ageyoungestchild` (that one's only defined for employees with children, so it doesn't behave like the others).
3. Compare the effect estimates and their standard errors across your two specifications (comparison table), and explain what you find.

## Question 6 [\*]

Consider the sample-size formula from the workshop (or `statsmodels`' `NormalIndPower`, either is fine) and the **Fine-Tuning Social Media Advertising** case study (`ab` – real click-through data for two ad versions).

1. Using Action A's actual observed click-through rate as your baseline assumption, calculate the expected number of impressions needed per arm to detect a **10% relative lift** at 80% power.
2. Recalculate the expected number by altering your assumptions – try a bigger lift (e.g. 50%), and a smaller one (e.g. 5%). How sensitive is the required sample size to the size of the effect you're trying to detect?